In [1]:
from pathlib import Path
import pandas as pd


# ============================================================
# Configuration
# ============================================================

BASE = Path.cwd().parent
RECIPE_DB = BASE / "dataset" / "RecipeDB_full"

recipes = pd.read_csv(
    RECIPE_DB / "recipes.csv"
)


print("=" * 70)
print("CELL 1 — RECIPE-LEVEL ANALYSIS")
print("=" * 70)


# ============================================================
# Basic structure
# ============================================================

print(f"\nShape: {recipes.shape}")

print(
    f"Total recipe records: "
    f"{len(recipes):,}"
)

print(
    f"Unique recipe IDs: "
    f"{recipes['recipe_id'].nunique():,}"
)

print(
    f"Missing recipe IDs: "
    f"{recipes['recipe_id'].isna().sum():,}"
)

print(
    f"Duplicate recipe IDs: "
    f"{recipes['recipe_id'].duplicated().sum():,}"
)


# ============================================================
# Titles
# ============================================================

print(
    f"\nUnique recipe titles: "
    f"{recipes['recipe_title'].nunique():,}"
)

print(
    f"Missing recipe titles: "
    f"{recipes['recipe_title'].isna().sum():,}"
)


# ============================================================
# Columns / dtypes
# ============================================================

print("\nColumns:")
print(recipes.columns.tolist())


print("\nData types:")
print(recipes.dtypes)


# ============================================================
# Missing / unique audit
# ============================================================

audit = pd.DataFrame({
    "missing": recipes.isna().sum(),
    "missing_pct": (
        recipes.isna().mean() * 100
    ).round(2),
    "unique": recipes.nunique(
        dropna=True
    )
})

print("\nMissing / unique audit:")
print(audit)


# ============================================================
# Sample
# ============================================================

print("\nFirst 5 recipes:")
print(
    recipes.head(5).to_string(
        index=False
    )
)

CELL 1 — RECIPE-LEVEL ANALYSIS

Shape: (16980, 5)
Total recipe records: 16,980
Unique recipe IDs: 16,980
Missing recipe IDs: 0
Duplicate recipe IDs: 0

Unique recipe titles: 16,747
Missing recipe titles: 0

Columns:
['recipe_id', 'recipe_title', 'url', 'image', 'overview']

Data types:
recipe_id        int64
recipe_title    object
url             object
image           object
overview        object
dtype: object

Missing / unique audit:
              missing  missing_pct  unique
recipe_id           0          0.0   16980
recipe_title        0          0.0   16747
url                 0          0.0   16980
image               0          0.0       1
overview            0          0.0     372

First 5 recipes:
 recipe_id                               recipe_title                                                            url                                                                     image                                                                                            o

In [2]:
# ============================================================
# CELL 2 — INGREDIENT ANALYSIS
# ============================================================

ingredients = pd.read_csv(
    RECIPE_DB / "recipe_ingredients.csv"
)


print("=" * 70)
print("CELL 2 — INGREDIENT ANALYSIS")
print("=" * 70)


required = [
    "recipe_id",
    "ingredient_name"
]

missing_required = [
    col for col in required
    if col not in ingredients.columns
]

if missing_required:
    raise ValueError(
        f"Missing required columns: "
        f"{missing_required}"
    )


# ============================================================
# Basic counts
# ============================================================

print(
    f"\nTotal ingredient rows: "
    f"{len(ingredients):,}"
)

print(
    f"Unique ingredient names: "
    f"{ingredients['ingredient_name'].nunique():,}"
)

print(
    f"Unique recipes represented: "
    f"{ingredients['recipe_id'].nunique():,}"
)


# ============================================================
# Ingredients per recipe
# ============================================================

ingredients_per_recipe = (
    ingredients
    .groupby("recipe_id")
    .size()
)


print("\nIngredients per recipe:")

print(
    f"Mean   : "
    f"{ingredients_per_recipe.mean():.2f}"
)

print(
    f"Median : "
    f"{ingredients_per_recipe.median():.2f}"
)

print(
    f"Min    : "
    f"{ingredients_per_recipe.min()}"
)

print(
    f"Max    : "
    f"{ingredients_per_recipe.max()}"
)


# ============================================================
# Ingredient frequency
# ============================================================

ingredient_frequency = (
    ingredients["ingredient_name"]
    .value_counts()
    .rename_axis("ingredient_name")
    .reset_index(
        name="occurrences"
    )
)


print("\nTop 30 ingredients:")

print(
    ingredient_frequency.head(30)
    .to_string(index=False)
)


# ============================================================
# Save
# ============================================================

OUT = (
    BASE
    / "EDA"
    / "results"
    / "recipe"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


ingredient_frequency.to_csv(
    OUT / "ingredient_frequency.csv",
    index=False
)


# ============================================================
# Missing / blank ingredient names
# ============================================================

missing_names = (
    ingredients["ingredient_name"]
    .isna()
    .sum()
)

blank_names = (
    ingredients["ingredient_name"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)


print(
    f"\nMissing ingredient names: "
    f"{missing_names:,}"
)

print(
    f"Blank ingredient names: "
    f"{blank_names:,}"
)

CELL 2 — INGREDIENT ANALYSIS

Total ingredient rows: 162,111
Unique ingredient names: 5,693
Unique recipes represented: 16,971

Ingredients per recipe:
Mean   : 9.55
Median : 9.00
Min    : 1
Max    : 45

Top 30 ingredients:
       ingredient_name  occurrences
                butter         7206
                  salt         6977
                 sugar         5003
                   egg         4862
             olive oil         3961
                 onion         3822
                  milk         3332
                 water         3287
          garlic clove         2770
                 flour         2487
          black pepper         2249
                pepper         1972
           lemon juice         1955
              cinnamon         1947
               parsley         1896
                tomato         1837
         purpose flour         1819
           salt pepper         1697
                garlic         1573
               oregano         1452
                 cre

In [3]:
# ============================================================
# CELL 3 — INGREDIENT NAME VARIATION
# ============================================================

import re


print("=" * 70)
print("CELL 3 — INGREDIENT NAME VARIATION")
print("=" * 70)


ingredient_names = (
    ingredients["ingredient_name"]
    .dropna()
    .astype(str)
)


# ============================================================
# Raw vs basic normalized names
# ============================================================

raw_unique = (
    ingredient_names
    .nunique()
)

normalized_names = (
    ingredient_names
    .str.strip()
    .str.lower()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

normalized_unique = (
    normalized_names
    .nunique()
)


print(
    f"Raw unique ingredient names: "
    f"{raw_unique:,}"
)

print(
    f"Lowercase/whitespace-normalized "
    f"unique names: {normalized_unique:,}"
)


# ============================================================
# Groups where normalization merges variants
# ============================================================

variation_df = pd.DataFrame({
    "raw_name": ingredient_names,
    "normalized_name": normalized_names
})


variant_groups = (
    variation_df
    .groupby("normalized_name")["raw_name"]
    .agg(
        number_of_variants="nunique",
        variants=lambda x: sorted(
            set(x)
        )
    )
    .reset_index()
)


variant_examples_df = (
    variant_groups[
        variant_groups[
            "number_of_variants"
        ] > 1
    ]
    .sort_values(
        "number_of_variants",
        ascending=False
    )
)


print(
    f"\nGroups with multiple raw variants "
    f"after basic normalization: "
    f"{len(variant_examples_df):,}"
)


if len(variant_examples_df) > 0:

    print(
        variant_examples_df
        .head(30)
        .to_string(index=False)
    )

else:

    print(
        "No multiple raw variants found "
        "using only lowercase + whitespace normalization."
    )


# ============================================================
# Common descriptive modifiers
# ============================================================

modifiers = [
    "fresh",
    "dried",
    "frozen",
    "ground",
    "minced",
    "chopped",
    "sliced",
    "diced",
    "crushed",
    "grated",
    "shredded",
    "roasted",
    "toasted",
    "cooked",
    "raw",
    "whole",
    "powdered",
    "organic",
    "boneless",
    "skinless"
]


modifier_counts = []

for modifier in modifiers:

    mask = (
        ingredient_names
        .str.lower()
        .str.contains(
            rf"\b{re.escape(modifier)}\b",
            regex=True,
            na=False
        )
    )

    modifier_counts.append({
        "modifier": modifier,
        "occurrences": int(mask.sum())
    })


modifier_df = (
    pd.DataFrame(modifier_counts)
    .sort_values(
        "occurrences",
        ascending=False
    )
)


print("\nDescriptive modifiers:")

print(
    modifier_df.to_string(
        index=False
    )
)


# ============================================================
# Common ingredient families
# ============================================================

keywords = [
    "garlic",
    "salt",
    "onion",
    "tomato",
    "pepper",
    "chicken",
    "beef",
    "mushroom",
    "butter",
    "olive oil"
]


for keyword in keywords:

    matches = (
        ingredient_names[
            ingredient_names
            .str.lower()
            .str.contains(
                keyword,
                na=False
            )
        ]
        .value_counts()
        .head(20)
    )

    print(
        f"\n--- Variants containing "
        f"'{keyword}' ---"
    )

    if len(matches) == 0:

        print("No matches.")

    else:

        print(matches.to_string())


# ============================================================
# Save
# ============================================================

variant_groups.to_csv(
    OUT / "ingredient_name_variation.csv",
    index=False
)

modifier_df.to_csv(
    OUT / "ingredient_modifiers.csv",
    index=False
)

CELL 3 — INGREDIENT NAME VARIATION
Raw unique ingredient names: 5,693
Lowercase/whitespace-normalized unique names: 5,693

Groups with multiple raw variants after basic normalization: 0
No multiple raw variants found using only lowercase + whitespace normalization.

Descriptive modifiers:
modifier  occurrences
 organic           16
   fresh            0
  frozen            0
   dried            0
  ground            0
  minced            0
  sliced            0
 chopped            0
 crushed            0
  grated            0
shredded            0
   diced            0
 roasted            0
 toasted            0
     raw            0
  cooked            0
   whole            0
powdered            0
boneless            0
skinless            0

--- Variants containing 'garlic' ---
ingredient_name
garlic clove                   2770
garlic                         1573
garlic powder                   396
garlic salt                      98
head garlic                      17
bulb garlic   

In [4]:
# ============================================================
# CELL 4 — QUANTITY / UNIT / STATE ANALYSIS
# ============================================================

print("=" * 70)
print("CELL 4 — QUANTITY / UNIT / STATE")
print("=" * 70)


required = [
    "ingredient_name",
    "quantity",
    "unit",
    "state"
]

missing_required = [
    col for col in required
    if col not in ingredients.columns
]

if missing_required:
    raise ValueError(
        f"Missing required columns: "
        f"{missing_required}"
    )


# ============================================================
# Missing values
# ============================================================

for col in required:

    missing = ingredients[col].isna().sum()

    blank = (
        ingredients[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(
        f"\n{col}"
    )

    print(
        f"  Missing: {missing:,} "
        f"({missing / len(ingredients) * 100:.2f}%)"
    )

    print(
        f"  Blank  : {blank:,} "
        f"({blank / len(ingredients) * 100:.2f}%)"
    )


# ============================================================
# Quantity numeric conversion
# ============================================================

quantity_numeric = pd.to_numeric(
    ingredients["quantity"],
    errors="coerce"
)


print("\nQuantity values:")

print(
    f"Numeric quantities: "
    f"{quantity_numeric.notna().sum():,}"
)

print(
    f"Non-numeric / missing quantities: "
    f"{quantity_numeric.isna().sum():,}"
)


print("\nNumeric quantity statistics:")

print(
    quantity_numeric
    .describe()
)


# ============================================================
# Units
# ============================================================

unit_counts = (
    ingredients["unit"]
    .fillna("")
    .astype(str)
    .str.strip()
)

unit_counts = (
    unit_counts[
        unit_counts != ""
    ]
    .value_counts()
    .rename_axis("unit")
    .reset_index(
        name="occurrences"
    )
)


print("\nMost common units:")

print(
    unit_counts.head(30)
    .to_string(index=False)
)


print("\nRare units (<= 5 occurrences):")

print(
    unit_counts[
        unit_counts["occurrences"] <= 5
    ]
    .to_string(index=False)
)


unit_counts.to_csv(
    OUT / "ingredient_units.csv",
    index=False
)


# ============================================================
# States
# ============================================================

state_counts = (
    ingredients["state"]
    .fillna("")
    .astype(str)
    .str.strip()
)

state_counts = (
    state_counts[
        state_counts != ""
    ]
    .value_counts()
    .rename_axis("state")
    .reset_index(
        name="occurrences"
    )
)


print("\nMost common states:")

print(
    state_counts.head(30)
    .to_string(index=False)
)


state_counts.to_csv(
    OUT / "ingredient_states.csv",
    index=False
)

CELL 4 — QUANTITY / UNIT / STATE

ingredient_name
  Missing: 177 (0.11%)
  Blank  : 177 (0.11%)

quantity
  Missing: 12,641 (7.80%)
  Blank  : 12,641 (7.80%)

unit
  Missing: 47,451 (29.27%)
  Blank  : 47,451 (29.27%)

state
  Missing: 100,469 (61.98%)
  Blank  : 100,469 (61.98%)

Quantity values:
Numeric quantities: 102,654
Non-numeric / missing quantities: 59,457

Numeric quantity statistics:
count    102654.000000
mean         13.031002
std          62.610021
min           0.125000
25%           1.000000
50%           2.000000
75%           3.000000
max        1600.000000
Name: quantity, dtype: float64

Most common units:
       unit  occurrences
        cup        28712
   teaspoon        20093
tablespoons        15000
       cups         9165
 tablespoon         8746
     ounces         5977
  teaspoons         4468
        1/2         2717
         lb         1820
        can         1809
     slices         1436
      pinch         1436
          g          992
    package      

In [6]:
# ============================================================
# CELL 5 — RECIPE COMPLEXITY
# ============================================================

print("=" * 70)
print("CELL 5 — RECIPE COMPLEXITY")
print("=" * 70)


ingredients_per_recipe = (
    ingredients
    .groupby("recipe_id")
    .size()
    .rename("ingredient_count")
    .reset_index()
)


print(
    f"\nRecipes represented: "
    f"{len(ingredients_per_recipe):,}"
)


print(
    f"Mean ingredients: "
    f"{ingredients_per_recipe['ingredient_count'].mean():.2f}"
)

print(
    f"Median ingredients: "
    f"{ingredients_per_recipe['ingredient_count'].median():.2f}"
)

print(
    f"Minimum ingredients: "
    f"{ingredients_per_recipe['ingredient_count'].min()}"
)

print(
    f"Maximum ingredients: "
    f"{ingredients_per_recipe['ingredient_count'].max()}"
)

print(
    f"Std deviation: "
    f"{ingredients_per_recipe['ingredient_count'].std():.2f}"
)


# ============================================================
# Percentiles
# ============================================================

print("\nIngredient count percentiles:")

print(
    ingredients_per_recipe[
        "ingredient_count"
    ]
    .quantile(
        [
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# ============================================================
# Frequency distribution
# ============================================================

complexity_distribution = (
    ingredients_per_recipe[
        "ingredient_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("ingredient_count")
    .reset_index(
        name="number_of_recipes"
    )
)


print("\nIngredient count distribution:")

print(
    complexity_distribution
    .head(50)
    .to_string(index=False)
)


# ============================================================
# Most complex recipes
# ============================================================

if "recipe_title" in recipes.columns:

    complex_with_titles = (
        ingredients_per_recipe
        .merge(
            recipes[
                [
                    "recipe_id",
                    "recipe_title"
                ]
            ],
            on="recipe_id",
            how="left"
        )
        .sort_values(
            "ingredient_count",
            ascending=False
        )
    )

    print("\n20 most complex recipes:")

    print(
        complex_with_titles
        .head(20)
        .to_string(index=False)
    )


# ============================================================
# Save
# ============================================================

ingredients_per_recipe.to_csv(
    OUT / "recipe_complexity.csv",
    index=False
)

complexity_distribution.to_csv(
    OUT / "recipe_complexity_distribution.csv",
    index=False
)

CELL 5 — RECIPE COMPLEXITY

Recipes represented: 16,971
Mean ingredients: 9.55
Median ingredients: 9.00
Minimum ingredients: 1
Maximum ingredients: 45
Std deviation: 4.09

Ingredient count percentiles:
0.01     3.0
0.05     4.0
0.25     7.0
0.50     9.0
0.75    12.0
0.90    15.0
0.95    17.0
0.99    22.0
Name: ingredient_count, dtype: float64

Ingredient count distribution:
 ingredient_count  number_of_recipes
                1                  3
                2                113
                3                366
                4                756
                5               1187
                6               1522
                7               1751
                8               1928
                9               1761
               10               1646
               11               1415
               12               1128
               13                863
               14                631
               15                545
               16              

In [7]:
# ============================================================
# CELL 6 — NUTRITION ANALYSIS + RELIABILITY AUDIT
# ============================================================

from pathlib import Path
import pandas as pd


BASE = Path.cwd().parent
RECIPE_DB = BASE / "dataset" / "RecipeDB_full"

EDA_RESULTS = (
    BASE
    / "EDA"
    / "results"
)

(EDA_RESULTS / "recipe").mkdir(
    parents=True,
    exist_ok=True
)


nutrition = pd.read_csv(
    RECIPE_DB / "recipe_nutrition.csv"
)


print("=" * 70)
print("CELL 6 — NUTRITION ANALYSIS")
print("=" * 70)


# ============================================================
# Basic structure
# ============================================================

print(
    f"\nShape: {nutrition.shape}"
)

print("\nColumns:")
print(
    nutrition.columns.tolist()
)

print("\nData types:")
print(
    nutrition.dtypes
)


# ============================================================
# Identify nutrient column
# ============================================================

nutrient_column = None

for candidate in [
    "nutrient",
    "nutrient_name"
]:

    if candidate in nutrition.columns:

        nutrient_column = candidate
        break


if nutrient_column is None:

    raise ValueError(
        "Could not identify nutrient-name column."
    )


print(
    f"\nNutrient column: "
    f"{nutrient_column}"
)


# ============================================================
# Basic counts
# ============================================================

print(
    f"\nTotal nutrition rows: "
    f"{len(nutrition):,}"
)

print(
    f"Unique recipes: "
    f"{nutrition['recipe_id'].nunique():,}"
)

print(
    f"Unique nutrients: "
    f"{nutrition[nutrient_column].nunique():,}"
)


print("\nNutrient frequencies:")

print(
    nutrition[nutrient_column]
    .value_counts()
    .head(50)
)


# ============================================================
# Nutrients per recipe
# ============================================================

nutrients_per_recipe = (
    nutrition
    .groupby("recipe_id")
    .size()
)


print("\nNutrients per recipe:")

print(
    nutrients_per_recipe.describe()
)


# ============================================================
# Missing audit
# ============================================================

nutrition_missing = pd.DataFrame({
    "missing": nutrition.isna().sum(),
    "missing_pct": (
        nutrition.isna().mean() * 100
    ).round(2)
})


print("\nMissing values:")

print(
    nutrition_missing
)


# ============================================================
# Identify value column
# ============================================================

value_column = None

for candidate in [
    "value",
    "amount",
    "quantity",
    "nutrient_value"
]:

    if candidate in nutrition.columns:

        value_column = candidate
        break


if value_column is None:

    print(
        "\nWARNING: Could not identify "
        "numeric nutrition value column."
    )

else:

    print(
        f"\nNutrition value column: "
        f"{value_column}"
    )

    nutrition[value_column] = pd.to_numeric(
        nutrition[value_column],
        errors="coerce"
    )


    # ========================================================
    # Major nutrient summaries
    # ========================================================

    major_patterns = {
        "energy": [
            "energy",
            "calorie"
        ],
        "protein": [
            "protein"
        ],
        "carbohydrate": [
            "carbohydrate",
            "carb"
        ],
        "fat": [
            "fat"
        ],
        "fiber": [
            "fiber",
            "fibre"
        ],
        "sugar": [
            "sugar"
        ],
        "sodium": [
            "sodium"
        ]
    }


    major_rows = []

    for label, patterns in major_patterns.items():

        mask = (
            nutrition[nutrient_column]
            .astype(str)
            .str.lower()
            .apply(
                lambda x: any(
                    pattern in x
                    for pattern in patterns
                )
            )
        )

        subset = nutrition.loc[
            mask,
            value_column
        ].dropna()

        if len(subset) == 0:
            continue

        major_rows.append({
            "nutrient_group": label,
            "rows": len(subset),
            "mean": subset.mean(),
            "median": subset.median(),
            "min": subset.min(),
            "max": subset.max()
        })


    major_summary = pd.DataFrame(
        major_rows
    )


    print("\nMajor nutrient summary:")

    print(
        major_summary.to_string(
            index=False
        )
    )


    # ========================================================
    # Repetition audit
    # ========================================================

    repetition = (
        nutrition
        .dropna(
            subset=[
                nutrient_column,
                value_column
            ]
        )
        .groupby(nutrient_column)[value_column]
        .agg(
            total_rows="size",
            unique_values="nunique"
        )
        .reset_index()
    )


    repetition["unique_value_ratio"] = (
        repetition["unique_values"]
        / repetition["total_rows"]
    )


    print(
        "\nNutrition repetition audit:"
    )

    print(
        repetition
        .sort_values(
            "unique_value_ratio"
        )
        .head(30)
        .to_string(index=False)
    )


    # ========================================================
    # Within-recipe repetition
    # ========================================================

    within_recipe = (
        nutrition
        .dropna(
            subset=[
                "recipe_id",
                value_column
            ]
        )
        .groupby("recipe_id")[value_column]
        .agg(
            rows="size",
            unique_values="nunique"
        )
        .reset_index()
    )


    within_recipe["all_values_identical"] = (
        within_recipe["unique_values"] == 1
    )


    print(
        "\nRecipes where all nutrition "
        "values are identical:"
    )

    print(
        within_recipe[
            "all_values_identical"
        ].value_counts()
    )


    print(
        "\nExamples:"
    )

    print(
        within_recipe[
            within_recipe[
                "all_values_identical"
            ]
        ]
        .head(20)
        .to_string(index=False)
    )


    # ========================================================
    # Save
    # ========================================================

    repetition.to_csv(
        EDA_RESULTS
        / "recipe"
        / "nutrition_repetition_audit.csv",
        index=False
    )


    major_summary.to_csv(
        EDA_RESULTS
        / "recipe"
        / "major_nutrient_summary.csv",
        index=False
    )


nutrition_missing.to_csv(
    EDA_RESULTS
    / "recipe"
    / "nutrition_missing_audit.csv"
)


print(
    "\nNutrition EDA complete."
)

CELL 6 — NUTRITION ANALYSIS

Shape: (2614920, 4)

Columns:
['recipe_id', 'recipe_title', 'nutrient', 'quantity']

Data types:
recipe_id        int64
recipe_title    object
nutrient        object
quantity        object
dtype: object

Nutrient column: nutrient

Total nutrition rows: 2,614,920
Unique recipes: 16,980
Unique nutrients: 153

Nutrient frequencies:
nutrient
Protein (g)                                                          33960
Total fats (g)                                                       16980
Energy (kCal)                                                        16980
Carbohydrates (g)                                                    16980
Fatty acids, total saturated (g)                                     16980
Fatty acids, total trans-polyenoic 20:3 n-6 (g)                      16980
Fructose (g)                                                         16980
Menaquinone-4 (g)                                                    16980
Tocotrienol, delta (mg)        

In [8]:
# ============================================================
# CELL 7 — RECIPE STEPS ANALYSIS
# ============================================================

from pathlib import Path
import pandas as pd


BASE = Path.cwd().parent
RECIPE_DB = BASE / "dataset" / "RecipeDB_full"

EDA_RESULTS = (
    BASE
    / "EDA"
    / "results"
    / "recipe"
)

EDA_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)


steps = pd.read_csv(
    RECIPE_DB / "recipe_steps.csv"
)

recipes_for_steps = pd.read_csv(
    RECIPE_DB / "recipes.csv"
)


print("=" * 70)
print("CELL 7 — RECIPE STEPS ANALYSIS")
print("=" * 70)


# ============================================================
# Basic structure
# ============================================================

print(
    f"\nShape: {steps.shape}"
)

print("\nColumns:")
print(
    steps.columns.tolist()
)

print("\nData types:")
print(
    steps.dtypes
)


# ============================================================
# Basic counts
# ============================================================

print(
    f"\nTotal step rows: "
    f"{len(steps):,}"
)

print(
    f"Unique recipes represented: "
    f"{steps['recipe_id'].nunique():,}"
)

print(
    f"Total recipes in recipes.csv: "
    f"{recipes_for_steps['recipe_id'].nunique():,}"
)


# ============================================================
# Steps per recipe
# ============================================================

steps_per_recipe = (
    steps
    .groupby("recipe_id")
    .size()
    .rename("step_count")
    .reset_index()
)


# Include recipes with zero steps

all_recipe_ids = (
    recipes_for_steps[
        ["recipe_id"]
    ]
    .drop_duplicates()
)


steps_per_recipe_full = (
    all_recipe_ids
    .merge(
        steps_per_recipe,
        on="recipe_id",
        how="left"
    )
)


steps_per_recipe_full["step_count"] = (
    steps_per_recipe_full["step_count"]
    .fillna(0)
    .astype(int)
)


# ============================================================
# Statistics
# ============================================================

print("\nSteps per recipe:")

print(
    steps_per_recipe_full[
        "step_count"
    ].describe()
)


zero_steps = (
    steps_per_recipe_full[
        "step_count"
    ] == 0
).sum()


one_step = (
    steps_per_recipe_full[
        "step_count"
    ] == 1
).sum()


multiple_steps = (
    steps_per_recipe_full[
        "step_count"
    ] > 1
).sum()


total_recipes = len(
    steps_per_recipe_full
)


print(
    f"\nZero-step recipes: "
    f"{zero_steps:,} "
    f"({zero_steps / total_recipes * 100:.2f}%)"
)

print(
    f"One-step recipes: "
    f"{one_step:,} "
    f"({one_step / total_recipes * 100:.2f}%)"
)

print(
    f"Multiple-step recipes: "
    f"{multiple_steps:,} "
    f"({multiple_steps / total_recipes * 100:.2f}%)"
)


# ============================================================
# Distribution
# ============================================================

step_count_distribution = (
    steps_per_recipe_full[
        "step_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("step_count")
    .reset_index(
        name="number_of_recipes"
    )
)


print("\nStep-count distribution:")

print(
    step_count_distribution
    .head(50)
    .to_string(index=False)
)


# ============================================================
# Save
# ============================================================

steps_per_recipe_full.to_csv(
    EDA_RESULTS
    / "recipe_step_counts.csv",
    index=False
)


step_count_distribution.to_csv(
    EDA_RESULTS
    / "step_count_distribution.csv",
    index=False
)


print("\nStep EDA complete.")

CELL 7 — RECIPE STEPS ANALYSIS

Shape: (16980, 3)

Columns:
['recipe_id', 'recipe_title', 'step']

Data types:
recipe_id        int64
recipe_title    object
step            object
dtype: object

Total step rows: 16,980
Unique recipes represented: 16,980
Total recipes in recipes.csv: 16,980

Steps per recipe:
count    16980.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: step_count, dtype: float64

Zero-step recipes: 0 (0.00%)
One-step recipes: 16,980 (100.00%)
Multiple-step recipes: 0 (0.00%)

Step-count distribution:
 step_count  number_of_recipes
          1              16980

Step EDA complete.


In [9]:
# ============================================================
# CELL 8 — PROCESSES AND UTENSILS AUDIT
# ============================================================

from pathlib import Path
import pandas as pd


BASE = Path.cwd().parent
RECIPE_DB = BASE / "dataset" / "RecipeDB_full"

EDA_RESULTS = (
    BASE
    / "EDA"
    / "results"
    / "recipe"
)

EDA_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 70)
print("CELL 8 — PROCESSES / UTENSILS AUDIT")
print("=" * 70)


def load_csv_safe(path):

    if not path.exists():

        return None, "FILE NOT FOUND"

    if path.stat().st_size == 0:

        return None, "EMPTY FILE"

    try:

        df = pd.read_csv(path)

        if len(df) == 0:

            return df, "NO ROWS"

        return df, "OK"

    except pd.errors.EmptyDataError:

        return None, "EMPTY FILE"

    except Exception as e:

        return None, f"ERROR: {e}"


processes, process_status = load_csv_safe(
    RECIPE_DB / "recipe_processes.csv"
)

utensils, utensil_status = load_csv_safe(
    RECIPE_DB / "recipe_utensils.csv"
)


# ============================================================
# Report
# ============================================================

print(
    f"\nProcesses file status: "
    f"{process_status}"
)

if processes is not None:

    print(
        f"Process rows: "
        f"{len(processes):,}"
    )

    print(
        f"Unique recipes: "
        f"{processes['recipe_id'].nunique():,}"
        if "recipe_id" in processes.columns
        else "recipe_id column unavailable"
    )

    print("\nColumns:")
    print(processes.columns.tolist())

    print("\nMissing values:")
    print(processes.isna().sum())


print(
    f"\nUtensils file status: "
    f"{utensil_status}"
)

if utensils is not None:

    print(
        f"Utensil rows: "
        f"{len(utensils):,}"
    )

    print(
        f"Unique recipes: "
        f"{utensils['recipe_id'].nunique():,}"
        if "recipe_id" in utensils.columns
        else "recipe_id column unavailable"
    )

    print("\nColumns:")
    print(utensils.columns.tolist())

    print("\nMissing values:")
    print(utensils.isna().sum())


# ============================================================
# Save audit
# ============================================================

audit = pd.DataFrame([
    {
        "table": "recipe_processes",
        "status": process_status,
        "rows": (
            len(processes)
            if processes is not None
            else 0
        )
    },
    {
        "table": "recipe_utensils",
        "status": utensil_status,
        "rows": (
            len(utensils)
            if utensils is not None
            else 0
        )
    }
])


audit.to_csv(
    EDA_RESULTS
    / "processes_utensils_audit.csv",
    index=False
)


print(
    "\nImportant:"
)

print(
    "An empty extracted table does NOT prove "
    "that the underlying RecipeDB has no "
    "process or utensil information."
)

CELL 8 — PROCESSES / UTENSILS AUDIT

Processes file status: EMPTY FILE

Utensils file status: EMPTY FILE

Important:
An empty extracted table does NOT prove that the underlying RecipeDB has no process or utensil information.


In [10]:
# ============================================================
# CELL 9 — RECIPE LINKS ANALYSIS
# ============================================================

from pathlib import Path
import pandas as pd


BASE = Path.cwd().parent
RECIPE_DB = BASE / "dataset" / "RecipeDB_full"

EDA_RESULTS = (
    BASE
    / "EDA"
    / "results"
    / "recipe"
)

EDA_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)


links = pd.read_csv(
    RECIPE_DB / "recipe_links.csv"
)

recipes_links = pd.read_csv(
    RECIPE_DB / "recipes.csv"
)


print("=" * 70)
print("CELL 9 — RECIPE LINKS ANALYSIS")
print("=" * 70)


# ============================================================
# Basic counts
# ============================================================

print(
    f"\nTotal link rows: "
    f"{len(links):,}"
)

print(
    f"Unique recipes with links: "
    f"{links['recipe_id'].nunique():,}"
)

print(
    f"Total recipes: "
    f"{recipes_links['recipe_id'].nunique():,}"
)


# ============================================================
# Links per recipe
# ============================================================

links_per_recipe = (
    links
    .groupby("recipe_id")
    .size()
    .rename("link_count")
    .reset_index()
)


print("\nLinks per recipe:")

print(
    links_per_recipe[
        "link_count"
    ].describe()
)


# ============================================================
# Exact duplicates
# ============================================================

print(
    f"\nExact duplicate rows: "
    f"{links.duplicated().sum():,}"
)


# ============================================================
# Identify URL column
# ============================================================

url_column = None

for candidate in [
    "url",
    "link",
    "recipe_url",
    "source_url"
]:

    if candidate in links.columns:

        url_column = candidate
        break


print(
    f"\nURL column: "
    f"{url_column}"
)


# ============================================================
# Duplicate recipe-URL pairs
# ============================================================

if url_column is not None:

    duplicate_pairs = (
        links
        .duplicated(
            subset=[
                "recipe_id",
                url_column
            ]
        )
        .sum()
    )

    print(
        f"Duplicate recipe-URL pairs: "
        f"{duplicate_pairs:,}"
    )


    missing_urls = (
        links[url_column]
        .isna()
        .sum()
    )

    blank_urls = (
        links[url_column]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(
        f"Missing URLs: "
        f"{missing_urls:,}"
    )

    print(
        f"Blank URLs: "
        f"{blank_urls:,}"
    )


# ============================================================
# Link count distribution
# ============================================================

link_count_distribution = (
    links_per_recipe[
        "link_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("link_count")
    .reset_index(
        name="number_of_recipes"
    )
)


print("\nLink-count distribution:")

print(
    link_count_distribution
    .to_string(index=False)
)


# ============================================================
# Save
# ============================================================

links_per_recipe.to_csv(
    EDA_RESULTS
    / "recipe_link_counts.csv",
    index=False
)

link_count_distribution.to_csv(
    EDA_RESULTS
    / "link_count_distribution.csv",
    index=False
)


print("\nLink EDA complete.")

CELL 9 — RECIPE LINKS ANALYSIS

Total link rows: 637,551
Unique recipes with links: 16,980
Total recipes: 16,980

Links per recipe:
count    16980.000000
mean        37.547173
std          4.098421
min         28.000000
25%         35.000000
50%         37.000000
75%         40.000000
max         73.000000
Name: link_count, dtype: float64

Exact duplicate rows: 108,365

URL column: url
Duplicate recipe-URL pairs: 142,325
Missing URLs: 0
Blank URLs: 0

Link-count distribution:
 link_count  number_of_recipes
         28                  9
         29                  3
         30                113
         31                366
         32                756
         33               1187
         34               1522
         35               1751
         36               1928
         37               1761
         38               1646
         39               1415
         40               1128
         41                863
         42                631
         43            

In [11]:
# ============================================================
# CELL 10 — INGREDIENT FREQUENCY + COVERAGE READINESS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


BASE = Path.cwd().parent

RECIPE_DB = (
    BASE
    / "dataset"
    / "RecipeDB_full"
)

EDA_RESULTS = (
    BASE
    / "EDA"
    / "results"
    / "recipe"
)

EDA_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)


ingredients = pd.read_csv(
    RECIPE_DB / "recipe_ingredients.csv"
)


print("=" * 70)
print("CELL 10 — INGREDIENT FREQUENCY + COVERAGE READINESS")
print("=" * 70)


# ============================================================
# Clean ingredient occurrence table
# ============================================================

ingredient_series = (
    ingredients["ingredient_name"]
    .dropna()
    .astype(str)
    .str.strip()
)

ingredient_series = ingredient_series[
    ingredient_series != ""
]


total_occurrences = len(
    ingredient_series
)

unique_ingredients = (
    ingredient_series.nunique()
)


print(
    f"\nTotal ingredient occurrences: "
    f"{total_occurrences:,}"
)

print(
    f"Unique ingredient names: "
    f"{unique_ingredients:,}"
)

print(
    f"Average occurrences per ingredient: "
    f"{total_occurrences / unique_ingredients:.2f}"
)


# ============================================================
# Frequency ranking
# ============================================================

frequency = (
    ingredient_series
    .value_counts()
    .rename_axis("ingredient_name")
    .reset_index(
        name="occurrences"
    )
)


frequency["rank"] = (
    np.arange(
        1,
        len(frequency) + 1
    )
)


frequency["cumulative_occurrences"] = (
    frequency["occurrences"]
    .cumsum()
)


frequency["cumulative_coverage"] = (
    frequency["cumulative_occurrences"]
    / total_occurrences
)


frequency["cumulative_coverage_pct"] = (
    frequency["cumulative_coverage"]
    * 100
)


# ============================================================
# Top-K coverage
# ============================================================

top_k_values = [
    10,
    25,
    50,
    100,
    250,
    500,
    1000,
    2500,
    5000,
    10000
]


top_k_rows = []

for k in top_k_values:

    if k > len(frequency):
        continue

    row = frequency.iloc[k - 1]

    top_k_rows.append({
        "top_k": k,
        "occurrences_covered": int(
            row["cumulative_occurrences"]
        ),
        "coverage_pct": float(
            row["cumulative_coverage_pct"]
        )
    })


top_k_coverage = pd.DataFrame(
    top_k_rows
)


print("\nTop-K occurrence coverage:")

print(
    top_k_coverage.to_string(
        index=False
    )
)


# ============================================================
# Coverage targets
# ============================================================

targets = [
    0.50,
    0.75,
    0.80,
    0.90,
    0.95,
    0.99
]


target_rows = []

for target in targets:

    idx = (
        frequency[
            "cumulative_coverage"
        ] >= target
    ).idxmax()

    row = frequency.loc[idx]

    target_rows.append({
        "target_coverage_pct": target * 100,
        "ingredient_types_required": int(
            row["rank"]
        ),
        "actual_coverage_pct": float(
            row["cumulative_coverage_pct"]
        )
    })


coverage_targets = pd.DataFrame(
    target_rows
)


print("\nIngredient types required for coverage targets:")

print(
    coverage_targets.to_string(
        index=False
    )
)


# ============================================================
# Frequency distribution
# ============================================================

frequency_distribution = (
    frequency["occurrences"]
    .value_counts()
    .sort_index()
    .rename_axis("occurrences_per_ingredient")
    .reset_index(
        name="number_of_ingredients"
    )
)


print(
    "\nFrequency distribution:"
)

print(
    frequency_distribution
    .head(50)
    .to_string(index=False)
)


# ============================================================
# Long-tail analysis
# ============================================================

rare_thresholds = [
    1,
    2,
    3,
    5,
    10
]


long_tail_rows = []

for threshold in rare_thresholds:

    mask = (
        frequency["occurrences"]
        <= threshold
    )

    ingredient_count = int(
        mask.sum()
    )

    occurrence_count = int(
        frequency.loc[
            mask,
            "occurrences"
        ].sum()
    )

    long_tail_rows.append({
        "max_occurrences": threshold,
        "ingredient_types": ingredient_count,
        "percentage_of_ingredient_types": (
            ingredient_count
            / unique_ingredients
            * 100
        ),
        "occurrences": occurrence_count,
        "percentage_of_occurrences": (
            occurrence_count
            / total_occurrences
            * 100
        )
    })


long_tail = pd.DataFrame(
    long_tail_rows
)


print("\nLong-tail summary:")

print(
    long_tail.to_string(
        index=False
    )
)


# ============================================================
# Singleton analysis
# ============================================================

singletons = frequency[
    frequency["occurrences"] == 1
]


print(
    f"\nSingleton ingredient types: "
    f"{len(singletons):,}"
)

print(
    f"Percentage of ingredient types "
    f"that occur once: "
    f"{len(singletons) / unique_ingredients * 100:.2f}%"
)

print(
    f"Percentage of all occurrences "
    f"represented by singletons: "
    f"{len(singletons) / total_occurrences * 100:.2f}%"
)


# ============================================================
# Top 30
# ============================================================

print("\nTop 30 ingredients:")

print(
    frequency
    .head(30)
    .to_string(index=False)
)


# ============================================================
# Save
# ============================================================

frequency.to_csv(
    EDA_RESULTS
    / "ingredient_frequency_ranked.csv",
    index=False
)

top_k_coverage.to_csv(
    EDA_RESULTS
    / "ingredient_top_k_coverage.csv",
    index=False
)

coverage_targets.to_csv(
    EDA_RESULTS
    / "ingredient_coverage_targets.csv",
    index=False
)

frequency_distribution.to_csv(
    EDA_RESULTS
    / "ingredient_frequency_distribution.csv",
    index=False
)

long_tail.to_csv(
    EDA_RESULTS
    / "ingredient_long_tail.csv",
    index=False
)


print(
    "\nCell 10 complete."
)

CELL 10 — INGREDIENT FREQUENCY + COVERAGE READINESS

Total ingredient occurrences: 161,934
Unique ingredient names: 5,693
Average occurrences per ingredient: 28.44

Top-K occurrence coverage:
 top_k  occurrences_covered  coverage_pct
    10                43707     26.990626
    25                68370     42.220905
    50                87928     54.298665
   100               108429     66.958761
   250               131925     81.468376
   500               144360     89.147430
  1000               152645     94.263712
  2500               158561     97.917053
  5000               161241     99.572048

Ingredient types required for coverage targets:
 target_coverage_pct  ingredient_types_required  actual_coverage_pct
                50.0                         39            50.045698
                75.0                        160            75.097879
                80.0                        225            80.043104
                90.0                        551            90.0

In [12]:
# ============================================================
# CELL 11 — RECIPE-LEVEL STRUCTURAL SANITY CHECK
# ============================================================

from pathlib import Path
import pandas as pd


BASE = Path.cwd().parent

RECIPE_DB = (
    BASE
    / "dataset"
    / "RecipeDB_full"
)

EDA_RESULTS = (
    BASE
    / "EDA"
    / "results"
    / "recipe"
)

EDA_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# Load tables
# ============================================================

recipes = pd.read_csv(
    RECIPE_DB / "recipes.csv"
)

ingredients = pd.read_csv(
    RECIPE_DB / "recipe_ingredients.csv"
)

nutrition = pd.read_csv(
    RECIPE_DB / "recipe_nutrition.csv"
)

steps = pd.read_csv(
    RECIPE_DB / "recipe_steps.csv"
)


print("=" * 70)
print("CELL 11 — RECIPE STRUCTURAL SANITY CHECK")
print("=" * 70)


# ============================================================
# Random sample
# ============================================================

sample_size = min(
    10,
    len(recipes)
)

sampled_recipes = (
    recipes[
        [
            "recipe_id",
            "recipe_title"
        ]
    ]
    .sample(
        n=sample_size,
        random_state=42
    )
)


print(
    f"\nSampling {sample_size} recipes."
)


# ============================================================
# Summary records
# ============================================================

summary_rows = []


for _, recipe in sampled_recipes.iterrows():

    recipe_id = recipe["recipe_id"]
    title = recipe["recipe_title"]


    recipe_ingredients = ingredients[
        ingredients["recipe_id"]
        == recipe_id
    ].copy()


    recipe_nutrition = nutrition[
        nutrition["recipe_id"]
        == recipe_id
    ].copy()


    recipe_steps = steps[
        steps["recipe_id"]
        == recipe_id
    ].copy()


    # --------------------------------------------------------
    # Ingredient checks
    # --------------------------------------------------------

    ingredient_count = len(
        recipe_ingredients
    )


    missing_ingredient_names = (
        recipe_ingredients[
            "ingredient_name"
        ]
        .isna()
        .sum()
    )


    blank_ingredient_names = (
        recipe_ingredients[
            "ingredient_name"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )


    missing_quantities = (
        recipe_ingredients[
            "quantity"
        ]
        .isna()
        .sum()
        if "quantity"
        in recipe_ingredients.columns
        else None
    )


    missing_units = (
        recipe_ingredients[
            "unit"
        ]
        .isna()
        .sum()
        if "unit"
        in recipe_ingredients.columns
        else None
    )


    # --------------------------------------------------------
    # Nutrition
    # --------------------------------------------------------

    nutrition_count = len(
        recipe_nutrition
    )


    # --------------------------------------------------------
    # Steps
    # --------------------------------------------------------

    step_count = len(
        recipe_steps
    )


    summary_rows.append({
        "recipe_id": recipe_id,
        "recipe_title": title,

        "ingredient_count":
            ingredient_count,

        "missing_ingredient_names":
            missing_ingredient_names,

        "blank_ingredient_names":
            blank_ingredient_names,

        "missing_quantities":
            missing_quantities,

        "missing_units":
            missing_units,

        "nutrition_rows":
            nutrition_count,

        "step_rows":
            step_count
    })


# ============================================================
# Summary table
# ============================================================

sanity_summary = pd.DataFrame(
    summary_rows
)


print("\nSample recipe summary:")

print(
    sanity_summary.to_string(
        index=False
    )
)


# ============================================================
# Reconstruct individual recipes
# ============================================================

for _, recipe in sampled_recipes.iterrows():

    recipe_id = recipe["recipe_id"]
    title = recipe["recipe_title"]


    recipe_ingredients = ingredients[
        ingredients["recipe_id"]
        == recipe_id
    ]


    recipe_nutrition = nutrition[
        nutrition["recipe_id"]
        == recipe_id
    ]


    recipe_steps = steps[
        steps["recipe_id"]
        == recipe_id
    ]


    print("\n")
    print("=" * 70)

    print(
        f"RECIPE {recipe_id}"
    )

    print(
        f"TITLE: {title}"
    )

    print("=" * 70)


    # --------------------------------------------------------
    # Ingredients
    # --------------------------------------------------------

    print("\nINGREDIENTS:")

    ingredient_columns = [
        col for col in [
            "ingredient_name",
            "quantity",
            "unit",
            "state"
        ]
        if col in recipe_ingredients.columns
    ]


    if len(recipe_ingredients) == 0:

        print("No ingredient rows found.")

    else:

        print(
            recipe_ingredients[
                ingredient_columns
            ]
            .to_string(index=False)
        )


    # --------------------------------------------------------
    # Nutrition
    # --------------------------------------------------------

    print("\nNUTRITION:")

    if len(recipe_nutrition) == 0:

        print("No nutrition rows found.")

    else:

        print(
            recipe_nutrition
            .head(20)
            .to_string(index=False)
        )


    # --------------------------------------------------------
    # Steps
    # --------------------------------------------------------

    print("\nSTEPS:")

    if len(recipe_steps) == 0:

        print("No step rows found.")

    else:

        print(
            recipe_steps
            .head(20)
            .to_string(index=False)
        )


# ============================================================
# Save
# ============================================================

sanity_summary.to_csv(
    EDA_RESULTS
    / "recipe_reconstruction_sanity_check.csv",
    index=False
)


print(
    "\nCell 11 complete."
)

CELL 11 — RECIPE STRUCTURAL SANITY CHECK

Sampling 10 recipes.

Sample recipe summary:
 recipe_id                         recipe_title  ingredient_count  missing_ingredient_names  blank_ingredient_names  missing_quantities  missing_units  nutrition_rows  step_rows
    108696       Baked Fish Plaka-Style (Greek)                 9                         0                       0                   1              5             154          1
    105082 Teisen Sir Fon (Anglesey Cake) Welsh                11                         0                       0                   0              1             154          1
    111013             Spinach Phyllo Casserole                 9                         0                       0                   0              1             154          1
    106047                        Swedish Glogg                11                         0                       0                   0              2             154          1
    103510            P

In [13]:
# ============================================================
# CELL 10 — INGREDIENT FREQUENCY + COVERAGE READINESS
# ============================================================

from pathlib import Path
import pandas as pd


# ============================================================
# Paths
# ============================================================

BASE = Path.cwd().parent

if not (BASE / "dataset").exists():
    BASE = BASE.parent

RECIPE_DB = BASE / "dataset" / "RecipeDB_full"
EDA_RESULTS = BASE / "EDA" / "results" / "recipe"

EDA_RESULTS.mkdir(parents=True, exist_ok=True)


# ============================================================
# Load ingredient data
# ============================================================

INGREDIENT_FILE = RECIPE_DB / "recipe_ingredients.csv"

if not INGREDIENT_FILE.exists():
    raise FileNotFoundError(
        f"Could not find:\n{INGREDIENT_FILE}"
    )

ingredients = pd.read_csv(INGREDIENT_FILE)


# ============================================================
# Basic validation
# ============================================================

required_columns = [
    "recipe_id",
    "ingredient_name"
]

missing_columns = [
    col for col in required_columns
    if col not in ingredients.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ============================================================
# Clean ingredient names only for this analysis
# ============================================================

ingredients_analysis = ingredients.copy()

ingredients_analysis["ingredient_name_clean"] = (
    ingredients_analysis["ingredient_name"]
    .astype("string")
    .str.strip()
)

ingredients_analysis = ingredients_analysis[
    ingredients_analysis["ingredient_name_clean"].notna()
    & (ingredients_analysis["ingredient_name_clean"] != "")
].copy()


# ============================================================
# Basic statistics
# ============================================================

total_occurrences = len(ingredients_analysis)

unique_ingredients = (
    ingredients_analysis["ingredient_name_clean"]
    .nunique()
)

average_occurrences = (
    total_occurrences / unique_ingredients
    if unique_ingredients > 0
    else 0
)

print("=" * 70)
print("CELL 10 — INGREDIENT FREQUENCY + COVERAGE READINESS")
print("=" * 70)

print(f"\nTotal ingredient occurrences : {total_occurrences:,}")
print(f"Unique ingredient names      : {unique_ingredients:,}")
print(f"Average occurrences / name  : {average_occurrences:.2f}")


# ============================================================
# Ingredient frequency ranking
# ============================================================

ingredient_frequency = (
    ingredients_analysis["ingredient_name_clean"]
    .value_counts()
    .rename_axis("ingredient_name")
    .reset_index(name="occurrences")
)

ingredient_frequency["rank"] = (
    ingredient_frequency.index + 1
)

ingredient_frequency["cumulative_occurrences"] = (
    ingredient_frequency["occurrences"]
    .cumsum()
)

ingredient_frequency["cumulative_coverage_pct"] = (
    ingredient_frequency["cumulative_occurrences"]
    / total_occurrences
    * 100
)

ingredient_frequency = ingredient_frequency[
    [
        "rank",
        "ingredient_name",
        "occurrences",
        "cumulative_occurrences",
        "cumulative_coverage_pct"
    ]
]


# ============================================================
# Top-K occurrence coverage
# ============================================================

top_k_values = [
    10,
    25,
    50,
    100,
    250,
    500,
    1000,
    2500,
    5000,
    10000
]

top_k_results = []

for k in top_k_values:

    if k <= len(ingredient_frequency):

        occurrences = ingredient_frequency.iloc[:k][
            "occurrences"
        ].sum()

        coverage = (
            occurrences / total_occurrences * 100
        )

        top_k_results.append({
            "top_k": k,
            "ingredient_types": k,
            "occurrences_covered": occurrences,
            "coverage_pct": coverage
        })


top_k_df = pd.DataFrame(top_k_results)

print("\nTop-K occurrence coverage:")
print(
    top_k_df.to_string(index=False)
)


# ============================================================
# Coverage targets
# ============================================================

coverage_targets = [
    50,
    75,
    80,
    90,
    95,
    99
]

target_results = []

for target in coverage_targets:

    matches = ingredient_frequency[
        ingredient_frequency["cumulative_coverage_pct"]
        >= target
    ]

    if len(matches) > 0:

        first_match = matches.iloc[0]

        target_results.append({
            "target_coverage_pct": target,
            "ingredient_types_required": int(
                first_match["rank"]
            ),
            "actual_coverage_pct": (
                first_match["cumulative_coverage_pct"]
            )
        })

    else:

        target_results.append({
            "target_coverage_pct": target,
            "ingredient_types_required": None,
            "actual_coverage_pct": None
        })


coverage_targets_df = pd.DataFrame(target_results)

print("\nIngredient types required for coverage targets:")
print(
    coverage_targets_df.to_string(index=False)
)


# ============================================================
# Frequency distribution
# ============================================================

frequency_distribution = (
    ingredient_frequency["occurrences"]
    .value_counts()
    .sort_index()
    .rename_axis("occurrence_count")
    .reset_index(name="number_of_ingredient_types")
)

print("\nIngredient frequency distribution:")
print(
    frequency_distribution.head(30).to_string(index=False)
)


# ============================================================
# Long-tail analysis
# ============================================================

rare_thresholds = [
    1,
    2,
    3,
    5,
    10
]

long_tail_results = []

for threshold in rare_thresholds:

    rare = ingredient_frequency[
        ingredient_frequency["occurrences"] <= threshold
    ]

    rare_types = len(rare)

    rare_occurrences = rare["occurrences"].sum()

    type_pct = (
        rare_types / unique_ingredients * 100
        if unique_ingredients > 0
        else 0
    )

    occurrence_pct = (
        rare_occurrences / total_occurrences * 100
        if total_occurrences > 0
        else 0
    )

    long_tail_results.append({
        "max_occurrences": threshold,
        "ingredient_types": rare_types,
        "ingredient_type_pct": type_pct,
        "occurrences": rare_occurrences,
        "occurrence_pct": occurrence_pct
    })


long_tail_df = pd.DataFrame(long_tail_results)

print("\nLong-tail analysis:")
print(
    long_tail_df.to_string(index=False)
)


# ============================================================
# Singleton ingredients
# ============================================================

singletons = ingredient_frequency[
    ingredient_frequency["occurrences"] == 1
]

print("\nSingleton ingredients:")
print(
    f"Unique ingredient types occurring exactly once: "
    f"{len(singletons):,}"
)

print(
    f"Percentage of ingredient types that are singletons: "
    f"{len(singletons) / unique_ingredients * 100:.2f}%"
)

print(
    f"Percentage of all ingredient occurrences contributed "
    f"by singletons: "
    f"{singletons['occurrences'].sum() / total_occurrences * 100:.2f}%"
)


# ============================================================
# Top 30 ingredients
# ============================================================

print("\nTop 30 ingredients by occurrence:")

print(
    ingredient_frequency
    .head(30)
    .to_string(index=False)
)


# ============================================================
# Save results
# ============================================================

ingredient_frequency.to_csv(
    EDA_RESULTS / "ingredient_frequency_ranked.csv",
    index=False
)

top_k_df.to_csv(
    EDA_RESULTS / "ingredient_top_k_coverage.csv",
    index=False
)

coverage_targets_df.to_csv(
    EDA_RESULTS / "ingredient_coverage_targets.csv",
    index=False
)

frequency_distribution.to_csv(
    EDA_RESULTS / "ingredient_frequency_distribution.csv",
    index=False
)

long_tail_df.to_csv(
    EDA_RESULTS / "ingredient_long_tail.csv",
    index=False
)


print("\n" + "=" * 70)
print("CELL 10 COMPLETE")
print("=" * 70)

print(f"\nResults saved to:")
print(EDA_RESULTS)

CELL 10 — INGREDIENT FREQUENCY + COVERAGE READINESS

Total ingredient occurrences : 161,934
Unique ingredient names      : 5,693
Average occurrences / name  : 28.44

Top-K occurrence coverage:
 top_k  ingredient_types  occurrences_covered  coverage_pct
    10                10                43707     26.990626
    25                25                68370     42.220905
    50                50                87928     54.298665
   100               100               108429     66.958761
   250               250               131925     81.468376
   500               500               144360     89.147430
  1000              1000               152645     94.263712
  2500              2500               158561     97.917053
  5000              5000               161241     99.572048

Ingredient types required for coverage targets:
 target_coverage_pct  ingredient_types_required  actual_coverage_pct
                  50                         39            50.045698
                  75

In [14]:
# ============================================================
# CELL 11 — RECIPE-LEVEL STRUCTURAL SANITY CHECK
# ============================================================

from pathlib import Path
import pandas as pd


# ============================================================
# Paths
# ============================================================

BASE = Path.cwd().parent

if not (BASE / "dataset").exists():
    BASE = BASE.parent

RECIPE_DB = BASE / "dataset" / "RecipeDB_full"
EDA_RESULTS = BASE / "EDA" / "results" / "recipe"

EDA_RESULTS.mkdir(parents=True, exist_ok=True)


# ============================================================
# Load datasets
# ============================================================

recipes_file = RECIPE_DB / "recipes.csv"
ingredients_file = RECIPE_DB / "recipe_ingredients.csv"
nutrition_file = RECIPE_DB / "recipe_nutrition.csv"
steps_file = RECIPE_DB / "recipe_steps.csv"


recipes = pd.read_csv(recipes_file)
ingredients = pd.read_csv(ingredients_file)
nutrition = pd.read_csv(nutrition_file)
steps = pd.read_csv(steps_file)


# ============================================================
# Validate required columns
# ============================================================

required_columns = {
    "recipes": ["recipe_id"],
    "ingredients": ["recipe_id"],
    "nutrition": ["recipe_id"],
    "steps": ["recipe_id"]
}

datasets = {
    "recipes": recipes,
    "ingredients": ingredients,
    "nutrition": nutrition,
    "steps": steps
}

for dataset_name, columns in required_columns.items():

    df = datasets[dataset_name]

    missing = [
        col for col in columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{dataset_name} is missing columns: {missing}"
        )


# ============================================================
# Random recipe sample
# ============================================================

SAMPLE_SIZE = 10
RANDOM_STATE = 42

available_recipe_ids = (
    recipes["recipe_id"]
    .dropna()
    .drop_duplicates()
)

sample_size = min(
    SAMPLE_SIZE,
    len(available_recipe_ids)
)

sample_ids = (
    available_recipe_ids
    .sample(
        n=sample_size,
        random_state=RANDOM_STATE
    )
    .tolist()
)


# ============================================================
# Helper function
# ============================================================

def is_missing_or_blank(series):

    if series is None:
        return True

    if len(series) == 0:
        return True

    return (
        series.isna().all()
        or series.astype("string").str.strip().eq("").all()
    )


# ============================================================
# Reconstruct sampled recipes
# ============================================================

summary_rows = []


print("=" * 70)
print("CELL 11 — RECIPE-LEVEL STRUCTURAL SANITY CHECK")
print("=" * 70)

print(
    f"\nRandomly sampled recipes: {sample_size}"
)


for recipe_id in sample_ids:

    # --------------------------------------------------------
    # Recipe metadata
    # --------------------------------------------------------

    recipe_rows = recipes[
        recipes["recipe_id"] == recipe_id
    ]

    title = None

    if "recipe_title" in recipe_rows.columns:
        if len(recipe_rows) > 0:
            title = recipe_rows.iloc[0]["recipe_title"]


    # --------------------------------------------------------
    # Ingredients
    # --------------------------------------------------------

    recipe_ingredients = ingredients[
        ingredients["recipe_id"] == recipe_id
    ].copy()

    ingredient_count = len(recipe_ingredients)

    missing_ingredient_names = 0

    if "ingredient_name" in recipe_ingredients.columns:

        missing_ingredient_names = (
            recipe_ingredients["ingredient_name"]
            .isna()
            .sum()
        )

        blank_ingredient_names = (
            recipe_ingredients["ingredient_name"]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )

        missing_ingredient_names += blank_ingredient_names


    # --------------------------------------------------------
    # Quantity / unit availability
    # --------------------------------------------------------

    missing_quantities = None
    missing_units = None

    if "quantity" in recipe_ingredients.columns:

        missing_quantities = (
            recipe_ingredients["quantity"]
            .isna()
            .sum()
        )

    if "unit" in recipe_ingredients.columns:

        missing_units = (
            recipe_ingredients["unit"]
            .isna()
            .sum()
        )


    # --------------------------------------------------------
    # Nutrition
    # --------------------------------------------------------

    recipe_nutrition = nutrition[
        nutrition["recipe_id"] == recipe_id
    ].copy()

    nutrition_rows = len(recipe_nutrition)

    nutrition_available = nutrition_rows > 0


    # --------------------------------------------------------
    # Steps
    # --------------------------------------------------------

    recipe_steps = steps[
        steps["recipe_id"] == recipe_id
    ].copy()

    step_count = len(recipe_steps)

    steps_available = step_count > 0


    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    summary_rows.append({

        "recipe_id": recipe_id,

        "recipe_title": title,

        "ingredient_count": ingredient_count,

        "missing_ingredient_names":
            missing_ingredient_names,

        "missing_quantities":
            missing_quantities,

        "missing_units":
            missing_units,

        "nutrition_rows":
            nutrition_rows,

        "nutrition_available":
            nutrition_available,

        "step_count":
            step_count,

        "steps_available":
            steps_available
    })


    # ========================================================
    # Print reconstructed recipe
    # ========================================================

    print("\n" + "-" * 70)

    print(f"Recipe ID : {recipe_id}")
    print(f"Title     : {title}")

    print("\nIngredients:")

    if len(recipe_ingredients) == 0:

        print("  [NO INGREDIENT RECORDS]")

    else:

        ingredient_columns = [
            col for col in [
                "ingredient_name",
                "quantity",
                "unit",
                "state"
            ]
            if col in recipe_ingredients.columns
        ]

        print(
            recipe_ingredients[
                ingredient_columns
            ]
            .to_string(index=False)
        )


    # --------------------------------------------------------
    # Nutrition
    # --------------------------------------------------------

    print("\nNutrition:")

    if len(recipe_nutrition) == 0:

        print("  [NO NUTRITION RECORDS]")

    else:

        print(
            recipe_nutrition
            .head(20)
            .to_string(index=False)
        )

        if len(recipe_nutrition) > 20:
            print(
                f"  ... "
                f"{len(recipe_nutrition) - 20} more rows"
            )


    # --------------------------------------------------------
    # Steps
    # --------------------------------------------------------

    print("\nSteps:")

    if len(recipe_steps) == 0:

        print("  [NO STEP RECORDS]")

    else:

        print(
            recipe_steps
            .head(20)
            .to_string(index=False)
        )

        if len(recipe_steps) > 20:
            print(
                f"  ... "
                f"{len(recipe_steps) - 20} more rows"
            )


# ============================================================
# Summary table
# ============================================================

sanity_check = pd.DataFrame(summary_rows)

print("\n" + "=" * 70)
print("STRUCTURAL SANITY SUMMARY")
print("=" * 70)

print(
    sanity_check.to_string(index=False)
)


# ============================================================
# Save summary
# ============================================================

output_file = (
    EDA_RESULTS /
    "recipe_reconstruction_sanity_check.csv"
)

sanity_check.to_csv(
    output_file,
    index=False
)


print("\n" + "=" * 70)
print("CELL 11 COMPLETE")
print("=" * 70)

print(f"\nSummary saved to:")
print(output_file)

CELL 11 — RECIPE-LEVEL STRUCTURAL SANITY CHECK

Randomly sampled recipes: 10

----------------------------------------------------------------------
Recipe ID : 108696
Title     : Baked Fish Plaka-Style (Greek)

Ingredients:
ingredient_name quantity        unit   state
          onion        4         NaN  sliced
      olive oil        3 tablespoons     NaN
         tomato        2         NaN     NaN
        parsley      1/2         cup chopped
   garlic clove        2         NaN  minced
          white    1 1/3         1/3     NaN
    lemon juice        2 tablespoons     NaN
    salt pepper      NaN         NaN     NaN
          lemon        1         NaN  sliced

Nutrition:
 recipe_id                   recipe_title                                        nutrient  quantity
    108696 Baked Fish Plaka-Style (Greek)                                  Total fats (g)   43.0128
    108696 Baked Fish Plaka-Style (Greek)                                   Energy (kCal)  894.7643
    108696 Ba